Code to generate the weather files to plot the IRS. 

We use one original weather file in the wofost format and create new .csvs with the data modified with the precipitation and temperature steps calculated in 02_ClimateScenariosProcess.ipynb



In [1]:
import pandas as pd
import numpy as np
import calendar

In [2]:
original_file = "data/metSensivity/era5_As_19900101_20241231.csv"
Tsteps_file = "methodology_outs/tas_change_steps_weighted_seasons_As.xlsx"
PPsteps_file = "methodology_outs/pr_change_steps_weighted_seasons_As.xlsx"
dir_out = "data/metSensivity/lagged/"
name_out = "era5_As_19900101_20241231"

In [3]:
PPsteps = pd.read_excel(PPsteps_file)
PPsteps

,Annual,DJF,MAM,JJA,SON
0,-15,1.14,-7.6,-30.08,-17.44
1,-10,6.14,-2.6,-25.08,-12.44
2,-5,11.14,2.4,-20.08,-7.44
3,0,0.00,0.0,0.00,0.00
4,5,21.14,12.4,-10.08,2.56
5,10,26.14,17.4,-5.08,7.56
6,15,31.14,22.4,-0.08,12.56
7,20,36.14,27.4,4.92,17.56
8,25,41.14,32.4,9.92,22.56


In [ ]:
##### 

# 1. Read full file as text

with open(original_file) as f:
    lines = f.readlines()

# Find the second @ line
start_line = [i for i, line in enumerate(lines) if line.startswith("@")][1]

# Extract column names (remove '@' and split)
columns = lines[start_line].replace("@", "").split()

# Read data, skipping the header line
weather = pd.read_csv(
    original_file,
    skiprows=start_line + 1,   # skip header + everything above
    sep=",", 
    names=columns
)
# 2. Get daily weather data only

weather["year"] = weather["DATE"] // 1000
weather["doy"] = weather["DATE"] % 1000



# 3. Get steps from excel file
Tsteps = pd.read_excel(Tsteps_file)
PPsteps = pd.read_excel(PPsteps_file)

# 4. Function to interpolate seasonal steps to daily values
season_days = {"DJF": 45, "MAM": 135, "JJA": 225, "SON": 315}

def interpolate_daily(step_row, year):
    ndays = 365 if not calendar.isleap(year) else 366
    x_days = np.arange(1, 366)  # always interpolate over 365
    yvals = [step_row["DJF"], step_row["MAM"], step_row["JJA"], step_row["SON"]]
    xvals = [season_days[s] for s in ["DJF","MAM","JJA","SON"]]
    daily_series = np.interp(x_days, xvals, yvals)
    if ndays == 366:
        daily_series = np.append(daily_series, daily_series[-1])  # repeat day 365
    return pd.Series(np.round(daily_series,2), index=np.arange(1, ndays+1))


#4. Apply Annual T and PP step

for Tstep in Tsteps["Annual"]:
    for PPstep in PPsteps["Annual"]:
        # 4a. Precipitation
        PPstep_row = PPsteps.loc[PPsteps["Annual"] == PPstep].iloc[0]

        weather_laggedPP = weather.copy()
        weather_laggedPP["PPanomaly"] = weather_laggedPP.apply(
            lambda r: interpolate_daily(PPstep_row, r["year"]).loc[r["doy"]],
            axis=1
        )
        weather_laggedPP["RAIN"] = weather_laggedPP["RAIN"] * (1 + weather_laggedPP["PPanomaly"] / 100.0)
        
        
        # 4b. Temperature
        Tstep_row = Tsteps.loc[Tsteps["Annual"] == Tstep].iloc[0]

        weather_laggedTandPP = weather_laggedPP.copy()
        weather_laggedTandPP["Tanomaly"] = weather_laggedTandPP.apply(
            lambda r: interpolate_daily(Tstep_row, r["year"]).loc[r["doy"]],
            axis=1
        )
        weather_laggedTandPP["TMIN"] = weather_laggedTandPP["TMIN"] + weather_laggedTandPP["Tanomaly"]
        weather_laggedTandPP["TMAX"] = weather_laggedTandPP["TMAX"] + weather_laggedTandPP["Tanomaly"]

        # 5. Ensure 8 columns and clean float artifacts
        expected_cols = ["DATE","SRAD","TMAX","TMIN","RAIN","RHUM"]
        for col in expected_cols:
            if col not in weather_laggedTandPP.columns:
                weather_laggedTandPP[col] = np.nan

        weather_final = weather_laggedTandPP[expected_cols].round(2)
        
        # 6. Write out full file again
        with open(f"{dir_out}{name_out}_Tstep_{Tstep}_PPstep_{PPstep}.csv", "w") as f:
            # Write back metadata lines unchanged
            for i in range(start_line):
                f.write(lines[i])
            # Write modified daily data
            weather_final.to_csv(f, index=False, na_rep="-99.0")
